Imports et configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Configuration visuelle
sns.set_theme(style="whitegrid")
%matplotlib inline

Chargement et nettoyage des données

In [2]:
# Chargement du dataset filtré (Angleterre uniquement)
df = pd.read_csv('England_football_data.csv', sep=';')

# Conversion de la date
df['Date'] = pd.to_datetime(df['Date'])

# Création de la variable cible : Total de buts dans le match
df['TotalGoals'] = df['FTHG'] + df['FTAG']

# Suppression des colonnes inutiles ou qui causent une fuite de données (scores mi-temps, etc.)
# On garde 'HomeTeam', 'AwayTeam' pour le Feature Engineering, on les traitera après.
cols_to_drop_later = ['id', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR']

print(f"Nombre de matchs chargés : {len(df)}")
df[['Date', 'HomeTeam', 'AwayTeam', 'TotalGoals']].head()

Nombre de matchs chargés : 10193


,Date,HomeTeam,AwayTeam,TotalGoals
0,2012-08-18,Arsenal,Sunderland,0.0
1,2012-08-18,Fulham,Norwich,5.0
2,2012-08-18,Newcastle,Tottenham,3.0
3,2012-08-18,QPR,Swansea,5.0
4,2012-08-18,Reading,Stoke,2.0


Feature Engineering : La "Forme" des équipes
On calcule la moyenne des 5 derniers matchs

In [ ]:
print("Calcul des indicateurs de forme (Moyennes glissantes sur 5 matchs)...")

#Trier par date pour respecter la chronologie
df = df.sort_values(by='Date')

# Restructurer les données pour avoir une ligne par équipe par match
df_home = df[['id', 'Date', 'HomeTeam', 'FTHG', 'FTAG']].rename(
    columns={'HomeTeam': 'Team', 'FTHG': 'GoalsScored', 'FTAG': 'GoalsConceded'}
)
df_away = df[['id', 'Date', 'AwayTeam', 'FTAG', 'FTHG']].rename(
    columns={'AwayTeam': 'Team', 'FTAG': 'GoalsScored', 'FTHG': 'GoalsConceded'}
)
df_stats = pd.concat([df_home, df_away]).sort_values(by=['Team', 'Date'])

#  Calculer la moyenne glissante (Rolling Window)
# IMPORTANT : on utilise .shift(1) pour ne pas inclure le match actuel dans la moyenne (Data Leakage)
N_MATCHS = 5

df_stats['AvgGoalsScored_Last5'] = df_stats.groupby('Team')['GoalsScored'].transform(
    lambda x: x.rolling(N_MATCHS, min_periods=1).mean().shift(1)
)
df_stats['AvgGoalsConceded_Last5'] = df_stats.groupby('Team')['GoalsConceded'].transform(
    lambda x: x.rolling(N_MATCHS, min_periods=1).mean().shift(1)
)

# Remplir les premiers matchs (NaN) par 0
df_stats = df_stats.fillna(0)

# Réintégrer ces stats dans le DataFrame principal
# Pour l'équipe à Domicile
df_stats_home = df_stats.rename(columns={
    'Team': 'HomeTeam',
    'AvgGoalsScored_Last5': 'Home_Avg_Goals_Last_5',
    'AvgGoalsConceded_Last5': 'Home_Avg_Conceded_Last_5'
})
# Pour l'équipe à l'Extérieur
df_stats_away = df_stats.rename(columns={
    'Team': 'AwayTeam',
    'AvgGoalsScored_Last5': 'Away_Avg_Goals_Last_5',
    'AvgGoalsConceded_Last5': 'Away_Avg_Conceded_Last_5'
})

# Merge sur l'ID du match
df = pd.merge(df, df_stats_home[['id', 'Home_Avg_Goals_Last_5', 'Home_Avg_Conceded_Last_5']], on='id', how='left')
df = pd.merge(df, df_stats_away[['id', 'Away_Avg_Goals_Last_5', 'Away_Avg_Conceded_Last_5']], on='id', how='left')

# Suppression des doublons éventuels
df = df.drop_duplicates(subset='id')

print("Feature Engineering terminé.")


Calcul des indicateurs de forme (Moyennes glissantes sur 5 matchs)...
Feature Engineering terminé.


,id,Country,League,Div,Season,Date,HomeTeam,AwayTeam,Referee,FTHG,...,Away_Avg_Goals_Last_5_x,Away_Avg_Conceded_Last_5_x,Home_Avg_Goals_Last_5_y,Home_Avg_Conceded_Last_5_y,Away_Avg_Goals_Last_5_y,Away_Avg_Conceded_Last_5_y,Home_Avg_Goals_Last_5,Home_Avg_Conceded_Last_5,Away_Avg_Goals_Last_5,Away_Avg_Conceded_Last_5
0,25,England,Premier League,E0,2012-2013,2012-01-09,Wigan,Stoke,M Atkinson,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,24,England,Premier League,E0,2012-2013,2012-01-09,West Ham,Fulham,A Taylor,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,23,England,Premier League,E0,2012-2013,2012-01-09,West Brom,Everton,J Moss,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,22,England,Premier League,E0,2012-2013,2012-01-09,Tottenham,Norwich,M Halsey,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16,21,England,Premier League,E0,2012-2013,2012-01-09,Swansea,Sunderland,R East,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Préparation pour le Machine Learning

In [ ]:
# 1. Nettoyage final des colonnes
X = df.drop(columns=cols_to_drop_later + ['Date', 'TotalGoals'], errors='ignore')
y = df['TotalGoals']

# 2. Identification automatique des types de colonnes
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Features Numériques ({len(numeric_features)}) : {numeric_features}")
print(f"Features Catégorielles ({len(categorical_features)}) : {categorical_features}")

# 3. Séparation Train / Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Création du Préprocesseur (ColumnTransformer)
# - Numérique : Imputation (médiane) + Standardisation
# - Catégoriel : Imputation (fréquent) + OneHotEncoding
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)